In [1]:
#Importar las librerias
from pathlib import Path
import sys

# Carpeta raíz del proyecto
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score, KFold, RandomizedSearchCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
import warnings
from tqdm import TqdmWarning 

warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../data/store_sales.csv')

In [3]:
# FEATURE ENGINEERING
customer_frequency = (
    df.groupby("CustomerID")
      .size()
)

df["Customer_Frequency"] = df["CustomerID"].map(customer_frequency)

In [4]:
df["Gender_Category"] = (
    df["Gender"] +
    "_" +
    df["Category"]
)

In [5]:
df["Payment_Category"] = (
    df["PaymentMethod"] +
    "_" +
    df["Category"]
)

In [6]:
bins = [18,25,35,45,60,100]

labels = [
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "60+"
]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels
)

In [7]:
df.drop(columns= ['Age', 'CustomerID'], inplace= True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Gender              5000 non-null   object  
 1   Category            5000 non-null   object  
 2   ItemPurchased       5000 non-null   object  
 3   Amount              5000 non-null   float64 
 4   Season              5000 non-null   object  
 5   PaymentMethod       5000 non-null   object  
 6   ItemRating          5000 non-null   float64 
 7   DiscountApplied(%)  5000 non-null   int64   
 8   PreviousPurchases   5000 non-null   int64   
 9   Customer_Frequency  5000 non-null   int64   
 10  Gender_Category     5000 non-null   object  
 11  Payment_Category    5000 non-null   object  
 12  Age_Group           5000 non-null   category
dtypes: category(1), float64(2), int64(3), object(7)
memory usage: 474.0+ KB


In [9]:
df['Age_Group'] = df['Age_Group'].astype('object')

In [10]:
X = df.drop('Amount', axis=1)
y = df['Amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=52)

In [11]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

numeric_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('poly', PolynomialFeatures(degree = 2, include_bias= False))
])

categoric_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categoric_pipeline, cat_cols)
])

selector = RFE(
    estimator = LinearRegression(),
    n_features_to_select= 10
)

In [12]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", selector),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

mae_lr_rfe = mean_absolute_error(y_test, y_pred)
mse_lr_rfe = mean_squared_error(y_test, y_pred)
rmse_lr_rfe = mse_lr_rfe ** 0.5
r2_lr_rfe = r2_score(y_test, y_pred)

print(f"MAE: {mae_lr_rfe}")
print(f"RMSE: {rmse_lr_rfe}")
print(f"R2: {r2_lr_rfe}")

MAE: 102.4252584083238
RMSE: 222.61127487511402
R2: 0.8325858510389685


In [13]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=0.01),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.01)
}

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "R²": r2_score(y_test, y_pred)
    })

In [14]:
results_df = pd.DataFrame(results)
results_df.loc[len(results_df)] = [
    "Linear Regression + RFE",
    mae_lr_rfe,
    rmse_lr_rfe,
    r2_lr_rfe
]

results_df

,Model,MAE,RMSE,R²
0,Linear Regression,92.578472,220.503843,0.835741
1,Ridge,92.540422,220.483277,0.835771
2,Lasso,91.809958,220.088016,0.836360
3,ElasticNet,91.913442,220.261972,0.836101
4,Linear Regression + RFE,102.425258,222.611275,0.832586


In [15]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1)
}

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=5,
        scoring="r2"
    )

    results.append({
        "Model": name,
        "Mean CV R2": scores.mean(),
        "Std CV": scores.std()
    })

cv_results = pd.DataFrame(results)

In [16]:
cv_results

,Model,Mean CV R2,Std CV
0,Linear,0.814567,0.021727
1,Ridge,0.814677,0.021635
2,Lasso,0.814874,0.021719
3,ElasticNet,0.801129,0.008655
